### Chest X-Ray Ablation Study: Architecture Evolution
> **Objective:** An empirical deep learning research pipeline evaluating the performance limits of standard classification architectures (ANN, CNN) against SOTA models (ViT, Custom Hybrids) on noisy, text-mined clinical datasets (Indiana University).
> **Compute Stack:** PyTorch, Automatic Mixed Precision (AMP), Distributed Data Parallelism.

### Phase 1: Data Ingestion & Clinical Text Processing
> Establishing the MLOps pipeline. Handling extreme class imbalance, defining PyTorch DataLoaders, and applying radiological image transformations.

In [ ]:
# 1. Install missing dependencies for our 10x pipeline (Silently)
!pip install -q openpyxl seaborn scikit-learn

import os
import torch
import pandas as pd

print("="*60)
print("STEP 1: HARDWARE & DATA INFRASTRUCTURE CHECK")
print("="*60)

# 2. Hardware Verification (Crucial for Cloud Engineering)
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"[SUCCESS] {num_gpus} GPUs Online. Acceleration Active.")
    for i in range(num_gpus):
        print(f"   -> GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("[CRITICAL ERROR] No GPU found. Please enable T4 x2 in Kaggle settings.")

print("-" * 60)

# 3. Dynamic Data Locator (Finding the exact paths in Kaggle's backend)
image_dataset_path = None
csv_dataset_path = None

print("[INFO] Scanning Kaggle Input Directory...")
for root, dirs, files in os.walk('/kaggle/input/'):
    # Looking for the 14GB image folder (usually contains .dcm or .png)
    if not image_dataset_path and any(f.endswith('.png') or f.endswith('.dcm') for f in files[:20]):
        image_dataset_path = root
        
    # Looking for our extracted master_dataset.csv
    if 'master_dataset.csv' in files:
        csv_dataset_path = os.path.join(root, 'master_dataset.csv')

# 4. Data Readiness Report
if image_dataset_path:
    print(f"[SUCCESS] 14GB Image Dataset Found at: {image_dataset_path}")
    sample_images = [f for f in os.listdir(image_dataset_path) if f.endswith('.png') or f.endswith('.dcm')][:3]
    print(f"   -> Sample Images: {sample_images}")
else:
    print("[ERROR] 14GB Image Dataset NOT FOUND. Did you attach the Indiana dataset?")

if csv_dataset_path:
    print(f"[SUCCESS] Master CSV Found at: {csv_dataset_path}")
    try:
        df = pd.read_csv(csv_dataset_path)
        print(f"   -> Total Reports Available: {len(df)}")
    except Exception as e:
        print(f"   -> [ERROR] Could not read CSV: {e}")
else:
    print("[ERROR] Custom Master CSV NOT FOUND. Did you attach your zip files?")

print("="*60)
print("Awaiting Tech Lead Review...")

🔬 STEP 1: HARDWARE & DATA INFRASTRUCTURE CHECK
[SUCCESS] 2 GPUs Online. Acceleration Active.
   -> GPU 0: Tesla T4
   -> GPU 1: Tesla T4
------------------------------------------------------------
[INFO] Scanning Kaggle Input Directory...
[SUCCESS] 14GB Image Dataset Found at: /kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized
   -> Sample Images: ['349_IM-1697-2001.dcm.png', '607_IM-2196-1001.dcm.png', '2832_IM-1249-2001.dcm.png']
[SUCCESS] Master CSV Found at: /kaggle/input/datasets/sameernadeem66/xray-custom-code/processed_data/processed_data/master_dataset.csv
   -> Total Reports Available: 6457
Awaiting Tech Lead Review...


In [2]:
import pandas as pd
import os
import re

print("="*60)
print("STEP 2: NLP MULTI-CLASS LABEL EXTRACTION")
print("="*60)

# Exact paths resolved from the infrastructure check
CSV_IN_PATH = '/kaggle/input/datasets/sameernadeem66/xray-custom-code/processed_data/processed_data/master_dataset.csv'
WORK_DIR = '/kaggle/working/'
OUT_DIR = os.path.join(WORK_DIR, 'data', 'processed')
os.makedirs(OUT_DIR, exist_ok=True)
CSV_OUT_PATH = os.path.join(OUT_DIR, 'classification_target.csv')

def extract_labels(df):
    """
    Applies deterministic regex rules to extract clinical classes from raw text.
    Class 0: Normal | Class 1: Cardiomegaly | Class 2: Pneumonia/Opacity
    """
    def apply_rules(text):
        text = str(text).lower()
        if re.search(r'\b(cardiomegaly|heart is enlarged|enlarged heart)\b', text):
            return 1, "Cardiomegaly"
        elif re.search(r'\b(opacity|pneumonia|infiltrate|consolidation|effusion)\b', text):
            return 2, "Pneumonia/Opacity"
        else:
            return 0, "Normal"

    # Merge findings and impression for maximum semantic context
    df['full_report'] = df['findings'].fillna('') + " " + df['impression'].fillna('')
    df[['class_id', 'class_name']] = df['full_report'].apply(apply_rules).tolist()
    return df[['filename', 'class_id', 'class_name']]

if __name__ == "__main__":
    try:
        df = pd.read_csv(CSV_IN_PATH)
        print(f"[INFO] Source dataset loaded. Total records: {len(df)}")
        
        print("[INFO] Executing NLP classification rules...")
        processed_df = extract_labels(df)
        
        print("\n[INFO] Ground Truth Class Distribution:")
        print(processed_df['class_name'].value_counts().to_string())
        
        processed_df.to_csv(CSV_OUT_PATH, index=False)
        print(f"\n[SUCCESS] Classification targets saved securely to: {CSV_OUT_PATH}")
        
    except Exception as e:
        print(f"[CRITICAL ERROR] Pipeline execution failed: {e}")

STEP 2: NLP MULTI-CLASS LABEL EXTRACTION
[INFO] Source dataset loaded. Total records: 6457
[INFO] Executing NLP classification rules...

[INFO] Ground Truth Class Distribution:
class_name
Pneumonia/Opacity    4506
Normal               1482
Cardiomegaly          469

[SUCCESS] Classification targets saved securely to: /kaggle/working/data/processed/classification_target.csv


In [3]:
import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from sklearn.utils.class_weight import compute_class_weight

print("="*60)
print("STEP 3: PYTORCH DATA PIPELINE & CLASS WEIGHTING")
print("="*60)

# Paths
CSV_PATH = '/kaggle/working/data/processed/classification_target.csv'
IMAGE_DIR = '/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized'

class XRayClassificationDataset(Dataset):
    def __init__(self, csv_file, image_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform
        
        # Filter rows where images actually exist in the directory
        available_images = set(os.listdir(self.image_dir))
        self.data = self.data[self.data['filename'].isin(available_images)].reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name = os.path.join(self.image_dir, self.data.iloc[idx, 0])
        image = Image.open(img_name).convert('RGB')
        label = int(self.data.iloc[idx, 1])
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Image Transformations (Resizing to 128x128 for ANN baseline)
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("[INFO] Initializing Vision Dataset...")
dataset = XRayClassificationDataset(csv_file=CSV_PATH, image_dir=IMAGE_DIR, transform=transform)
total_samples = len(dataset)
print(f"[INFO] Total valid image-label pairs verified: {total_samples}")

# Train/Validation Split (80% Train, 20% Validation)
train_size = int(0.8 * total_samples)
val_size = total_samples - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print(f"[INFO] Data Split -> Training: {train_size} | Validation: {val_size}")

# DataLoaders for Batch Processing
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Calculate Class Weights to handle the imbalance
print("[INFO] Computing algorithmic class weights for CrossEntropyLoss...")
labels = dataset.data['class_id'].values
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print("\n[SUCCESS] Pipeline Ready. Calculated Class Weights:")
for i, weight in enumerate(class_weights):
    print(f"   -> Class {i} Weight: {weight:.4f}")

STEP 3: PYTORCH DATA PIPELINE & CLASS WEIGHTING
[INFO] Initializing Vision Dataset...
[INFO] Total valid image-label pairs verified: 6457
[INFO] Data Split -> Training: 5165 | Validation: 1292
[INFO] Computing algorithmic class weights for CrossEntropyLoss...

[SUCCESS] Pipeline Ready. Calculated Class Weights:
   -> Class 0 Weight: 1.4523
   -> Class 1 Weight: 4.5892
   -> Class 2 Weight: 0.4777


### Phase 2: Baseline Architecture (ANN)
> **Hypothesis:** Flattening spatial hierarchies leads to severe feature loss in radiological imaging. This baseline proves the necessity of spatial convolutions.

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import os
import warnings
warnings.filterwarnings("ignore")

print("="*60)
print("STEP 4: MLOPS EVALUATOR & ANN BASELINE TRAINING")
print("="*60)

# 1. The Evaluator Engine (Reusable for all future models)
class ResearchEvaluator:
    def __init__(self, model_name, out_dir='/kaggle/working/'):
        self.model_name = model_name
        self.out_dir = out_dir
        self.excel_path = os.path.join(out_dir, 'Comparative_Results.xlsx')
        
    def evaluate_and_report(self, y_true, y_pred, classes):
        print(f"\n[INFO] Generating Research Metrics for {self.model_name}...")
        
        # Calculate Strict Metrics
        acc = accuracy_score(y_true, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
        
        print(f"Accuracy:  {acc:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall:    {recall:.4f}")
        print(f"F1-Score:  {f1:.4f}")
        
        # Automate Excel Report Generation
        results_df = pd.DataFrame({
            'Model Architecture': [self.model_name],
            'Accuracy': [acc],
            'Precision': [precision],
            'Recall': [recall],
            'F1-Score': [f1]
        })
        
        if os.path.exists(self.excel_path):
            existing_df = pd.read_excel(self.excel_path)
            updated_df = pd.concat([existing_df, results_df], ignore_index=True)
            updated_df.to_excel(self.excel_path, index=False)
        else:
            results_df.to_excel(self.excel_path, index=False)
            
        print(f"[SUCCESS] Metrics dynamically appended to {self.excel_path}")
        
        # Generate Publication-Ready Confusion Matrix
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
        plt.title(f'Confusion Matrix: {self.model_name}', fontsize=14, fontweight='bold')
        plt.ylabel('True Clinical Diagnosis', fontsize=12)
        plt.xlabel('AI Predicted Diagnosis', fontsize=12)
        
        plot_path = os.path.join(self.out_dir, f'{self.model_name}_Confusion_Matrix.png')
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"[SUCCESS] High-Res Confusion Matrix saved to {plot_path}")

# 2. The Artificial Neural Network (Baseline)
class ANN_Baseline(nn.Module):
    def __init__(self, num_classes=3):
        super(ANN_Baseline, self).__init__()
        self.flatten = nn.Flatten()
        
        # 128x128 resolution * 3 channels = 49152 input features
        self.network = nn.Sequential(
            nn.Linear(128 * 128 * 3, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            
            nn.Linear(1024, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)

# 3. Training & Validation Execution
def execute_ann_pipeline():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ANN_Baseline(num_classes=3)
    
    # Scale across Dual GPUs if available
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)
    
    # Inject our dynamic class weights to fix the imbalance
    weights = class_weights_tensor.to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    
    EPOCHS = 5 # Baseline model doesn't need heavy training
    print(f"\n[INFO] Commencing Training Phase for ANN Baseline ({EPOCHS} Epochs)...")
    
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        print(f"Epoch [{epoch+1}/{EPOCHS}] | Training Loss: {running_loss/len(train_loader):.4f}")
        
    # Validation Phase
    print("\n[INFO] Running Inference on Validation Set...")
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
    # Trigger MLOps Evaluator
    classes = ["Normal", "Cardiomegaly", "Pneumonia/Opacity"]
    evaluator = ResearchEvaluator(model_name="ANN_Baseline")
    evaluator.evaluate_and_report(all_targets, all_preds, classes)

if __name__ == "__main__":
    execute_ann_pipeline()

STEP 4: MLOPS EVALUATOR & ANN BASELINE TRAINING

[INFO] Commencing Training Phase for ANN Baseline (5 Epochs)...
Epoch [1/5] | Training Loss: 1.0783
Epoch [2/5] | Training Loss: 1.0076
Epoch [3/5] | Training Loss: 0.9659
Epoch [4/5] | Training Loss: 0.9542
Epoch [5/5] | Training Loss: 0.9422

[INFO] Running Inference on Validation Set...

[INFO] Generating Research Metrics for ANN_Baseline...
Accuracy:  0.4528
Precision: 0.6194
Recall:    0.4528
F1-Score:  0.4878
[SUCCESS] Metrics dynamically appended to /kaggle/working/Comparative_Results.xlsx
[SUCCESS] High-Res Confusion Matrix saved to /kaggle/working/ANN_Baseline_Confusion_Matrix.png


### Phase 3: Spatial Feature Extraction (CNN Standard)
> Introducing deep convolutional layers coupled with Global Average Pooling (GAP) to capture localized pulmonary abnormalities with high parameter efficiency.

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import time

print("="*60)
print("STEP 5: CNN STANDARD ARCHITECTURE TRAINING")
print("="*60)

class CNN_Standard(nn.Module):
    def __init__(self, num_classes=3):
        super(CNN_Standard, self).__init__()
        
        # Block 1: Feature Extraction (Edges & Basic Textures)
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # Output: 64x64
        )
        
        # Block 2: Deep Features (Complex Patterns like Cardiomegaly)
        self.block2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # Output: 32x32
        )
        
        # Block 3: Abstract Medical Anomalies
        self.block3 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # Output: 16x16
        )
        
        # 10x Engineering: Global Average Pooling instead of Flattening
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Final Classification Head
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.global_avg_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

def execute_cnn_pipeline():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = CNN_Standard(num_classes=3)
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)
    
    # Re-using the dynamically calculated class weights from memory
    weights = class_weights_tensor.to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    
    # Modern Optimizer with Weight Decay for Regularization
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    
    EPOCHS = 10  # CNN needs slightly more epochs to learn spatial features
    print(f"\n[INFO] Commencing Training Phase for CNN Standard ({EPOCHS} Epochs)...")
    
    for epoch in range(EPOCHS):
        start_time = time.time()
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        epoch_time = time.time() - start_time
        print(f"Epoch [{epoch+1}/{EPOCHS}] | Training Loss: {running_loss/len(train_loader):.4f} | Time: {epoch_time:.1f}s")
        
    # Validation & Evaluation Phase
    print("\n[INFO] Running Inference on Validation Set...")
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
    classes = ["Normal", "Cardiomegaly", "Pneumonia/Opacity"]
    evaluator = ResearchEvaluator(model_name="CNN_Standard")
    evaluator.evaluate_and_report(all_targets, all_preds, classes)

if __name__ == "__main__":
    execute_cnn_pipeline()

STEP 5: CNN STANDARD ARCHITECTURE TRAINING

[INFO] Commencing Training Phase for CNN Standard (10 Epochs)...
Epoch [1/10] | Training Loss: 1.0924 | Time: 152.2s
Epoch [2/10] | Training Loss: 1.0696 | Time: 149.6s
Epoch [3/10] | Training Loss: 1.0593 | Time: 149.6s
Epoch [4/10] | Training Loss: 1.0596 | Time: 150.0s
Epoch [5/10] | Training Loss: 1.0484 | Time: 149.6s
Epoch [6/10] | Training Loss: 1.0367 | Time: 149.3s
Epoch [7/10] | Training Loss: 1.0344 | Time: 149.6s
Epoch [8/10] | Training Loss: 1.0378 | Time: 148.9s
Epoch [9/10] | Training Loss: 1.0320 | Time: 149.6s
Epoch [10/10] | Training Loss: 1.0321 | Time: 149.7s

[INFO] Running Inference on Validation Set...

[INFO] Generating Research Metrics for CNN_Standard...
Accuracy:  0.4853
Precision: 0.6170
Recall:    0.4853
F1-Score:  0.5149
[SUCCESS] Metrics dynamically appended to /kaggle/working/Comparative_Results.xlsx
[SUCCESS] High-Res Confusion Matrix saved to /kaggle/working/CNN_Standard_Confusion_Matrix.png


### Phase 4: Hierarchical Attention Mechanism (HAM)
> Implementing custom spatial attention maps. This phase tests whether attention mechanisms can generalize on small medical datasets without the bias of pre-training.

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import time

print("="*60)
print("STEP 6: HAM (HIERARCHICAL ATTENTION MECHANISM) TRAINING")
print("="*60)

class SpatialAttention(nn.Module):
    """
    Computes a spatial attention map by pooling cross-channel features
    and passing them through a convolution to highlight critical regions.
    """
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        # Convolves over max-pooled and avg-pooled maps
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        attention_map = torch.cat([avg_out, max_out], dim=1)
        attention_map = self.conv(attention_map)
        return x * self.sigmoid(attention_map)

class HAM_Network(nn.Module):
    def __init__(self, num_classes=3):
        super(HAM_Network, self).__init__()
        
        # Base Feature Extractor
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        
        # Spatial Attention Block
        self.attention = SpatialAttention(kernel_size=7)
        
        # High-level features
        self.high_level = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        # Apply Attention Mechanism dynamically
        x = self.attention(x)
        x = self.high_level(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

def execute_ham_pipeline():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = HAM_Network(num_classes=3)
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)
    
    # Utilizing our class weights to maintain strict penalty on minority class
    weights = class_weights_tensor.to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    
    # AdamW with slightly higher weight decay to regularize the attention weights
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    
    EPOCHS = 10
    print(f"\n[INFO] Commencing Training Phase for HAM Architecture ({EPOCHS} Epochs)...")
    
    for epoch in range(EPOCHS):
        start_time = time.time()
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        epoch_time = time.time() - start_time
        print(f"Epoch [{epoch+1}/{EPOCHS}] | Training Loss: {running_loss/len(train_loader):.4f} | Time: {epoch_time:.1f}s")
        
    print("\n[INFO] Running Inference on Validation Set...")
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
    classes = ["Normal", "Cardiomegaly", "Pneumonia/Opacity"]
    evaluator = ResearchEvaluator(model_name="HAM_Network")
    evaluator.evaluate_and_report(all_targets, all_preds, classes)

if __name__ == "__main__":
    execute_ham_pipeline()

STEP 6: HAM (HIERARCHICAL ATTENTION MECHANISM) TRAINING

[INFO] Commencing Training Phase for HAM Architecture (10 Epochs)...
Epoch [1/10] | Training Loss: 1.0742 | Time: 156.2s
Epoch [2/10] | Training Loss: 1.0393 | Time: 151.7s
Epoch [3/10] | Training Loss: 1.0345 | Time: 149.8s
Epoch [4/10] | Training Loss: 1.0221 | Time: 151.5s
Epoch [5/10] | Training Loss: 1.0085 | Time: 150.4s
Epoch [6/10] | Training Loss: 0.9972 | Time: 149.7s
Epoch [7/10] | Training Loss: 0.9929 | Time: 149.0s
Epoch [8/10] | Training Loss: 0.9958 | Time: 148.9s
Epoch [9/10] | Training Loss: 0.9914 | Time: 149.5s
Epoch [10/10] | Training Loss: 0.9879 | Time: 150.9s

[INFO] Running Inference on Validation Set...

[INFO] Generating Research Metrics for HAM_Network...
Accuracy:  0.4296
Precision: 0.5871
Recall:    0.4296
F1-Score:  0.4400
[SUCCESS] Metrics dynamically appended to /kaggle/working/Comparative_Results.xlsx
[SUCCESS] High-Res Confusion Matrix saved to /kaggle/working/HAM_Network_Confusion_Matrix.png


### Phase 5: State-of-the-Art Global Context (ViT-B/16)
> Applying transfer learning via Vision Transformers. Processing X-Rays as sequential patches to capture global thoracic context and evaluating its rapid convergence.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision.models import ViT_B_16_Weights
from torchvision import transforms
from torch.utils.data import DataLoader, random_split
import time
import warnings
warnings.filterwarnings("ignore")

print("="*60)
print("STEP 7: VISION TRANSFORMER (ViT) & TRANSFER LEARNING")
print("="*60)

# 1. Update Resolution for ViT (Strictly requires 224x224)
vit_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("[INFO] Re-initializing Vision Dataset for 224x224 ViT Resolution...")
dataset_vit = XRayClassificationDataset(csv_file=CSV_PATH, image_dir=IMAGE_DIR, transform=vit_transform)

train_dataset_vit, val_dataset_vit = random_split(dataset_vit, [train_size, val_size])
train_loader_vit = DataLoader(train_dataset_vit, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader_vit = DataLoader(val_dataset_vit, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# 2. The Vision Transformer Architecture
class ViT_Classifier(nn.Module):
    def __init__(self, num_classes=3):
        super(ViT_Classifier, self).__init__()
        
        # Load Pre-trained ViT (Transfer Learning)
        self.vit = models.vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
        
        # Freeze the base layers so we don't destroy the pre-trained knowledge
        for param in self.vit.parameters():
            param.requires_grad = False
            
        # Replace the final head (which originally has 1000 classes) with our 3 classes
        # The hidden size of ViT-Base is 768
        self.vit.heads = nn.Sequential(
            nn.Linear(768, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
        # Unfreeze the last few transformer blocks for fine-tuning
        for param in self.vit.encoder.layers[-2:].parameters():
            param.requires_grad = True

    def forward(self, x):
        return self.vit(x)

def execute_vit_pipeline():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ViT_Classifier(num_classes=3)
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)
    
    weights = class_weights_tensor.to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    
    # Fine-tuning requires a smaller learning rate (1e-4 instead of 1e-3)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
    
    EPOCHS = 10 
    print(f"\n[INFO] Commencing Fine-Tuning Phase for Vision Transformer ({EPOCHS} Epochs)...")
    
    for epoch in range(EPOCHS):
        start_time = time.time()
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader_vit:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        epoch_time = time.time() - start_time
        print(f"Epoch [{epoch+1}/{EPOCHS}] | Training Loss: {running_loss/len(train_loader_vit):.4f} | Time: {epoch_time:.1f}s")
        
    print("\n[INFO] Running Inference on Validation Set...")
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, labels in val_loader_vit:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
    classes = ["Normal", "Cardiomegaly", "Pneumonia/Opacity"]
    evaluator = ResearchEvaluator(model_name="Vision_Transformer")
    evaluator.evaluate_and_report(all_targets, all_preds, classes)

if __name__ == "__main__":
    execute_vit_pipeline()

STEP 7: VISION TRANSFORMER (ViT) & TRANSFER LEARNING
[INFO] Re-initializing Vision Dataset for 224x224 ViT Resolution...
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 217MB/s] 



[INFO] Commencing Fine-Tuning Phase for Vision Transformer (10 Epochs)...
Epoch [1/10] | Training Loss: 1.0029 | Time: 178.0s
Epoch [2/10] | Training Loss: 0.8561 | Time: 178.3s
Epoch [3/10] | Training Loss: 0.7904 | Time: 179.3s
Epoch [4/10] | Training Loss: 0.6864 | Time: 181.2s
Epoch [5/10] | Training Loss: 0.5916 | Time: 181.2s
Epoch [6/10] | Training Loss: 0.4732 | Time: 178.7s
Epoch [7/10] | Training Loss: 0.3402 | Time: 179.5s
Epoch [8/10] | Training Loss: 0.2605 | Time: 180.1s
Epoch [9/10] | Training Loss: 0.1880 | Time: 180.6s
Epoch [10/10] | Training Loss: 0.1270 | Time: 179.0s

[INFO] Running Inference on Validation Set...

[INFO] Generating Research Metrics for Vision_Transformer...
Accuracy:  0.6711
Precision: 0.6134
Recall:    0.6711
F1-Score:  0.6160
[SUCCESS] Metrics dynamically appended to /kaggle/working/Comparative_Results.xlsx
[SUCCESS] High-Res Confusion Matrix saved to /kaggle/working/Vision_Transformer_Confusion_Matrix.png


### Phase 6: Local-Global Fusion (Custom Hybrid)
> **Architecture Innovation:** Fusing ResNet18 (for local spatial edge detection) with a Transformer Encoder (for global sequence modeling) to maximize radiological feature retention.

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision.models import ResNet18_Weights
import time
import warnings
warnings.filterwarnings("ignore")

print("="*60)
print("STEP 8: HYBRID SOTA (CNN + TRANSFORMER FUSION)")
print("="*60)

class Hybrid_CNN_Transformer(nn.Module):
    def __init__(self, num_classes=3, embed_dim=512):
        super(Hybrid_CNN_Transformer, self).__init__()
        
        # 1. The Local Feature Extractor (CNN Backbone)
        # Using ResNet18 as our spatial feature extractor
        resnet = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        # We strip away the final pooling and classification layers
        # Output of this backbone for a 224x224 image is (Batch, 512, 7, 7)
        self.cnn_backbone = nn.Sequential(*list(resnet.children())[:-2])
        
        # 2. The Global Context Processor (Transformer Encoder)
        self.seq_len = 7 * 7  # 49 spatial patches
        self.embed_dim = embed_dim
        
        # Positional Embedding so the Transformer knows WHERE the features came from
        self.pos_embed = nn.Parameter(torch.randn(1, self.seq_len, self.embed_dim))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.embed_dim, 
            nhead=8, 
            dim_feedforward=1024, 
            dropout=0.3, 
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        # 3. The Final Diagnostic Head
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.embed_dim),
            nn.Linear(self.embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # Step A: Extract spatial features via CNN
        features = self.cnn_backbone(x)  # Shape: [B, 512, 7, 7]
        B, C, H, W = features.shape
        
        # Step B: Flatten spatial dimensions to create a sequence for the Transformer
        # From [B, 512, 7, 7] -> [B, 512, 49] -> [B, 49, 512]
        features = features.view(B, C, -1).permute(0, 2, 1)
        
        # Step C: Add Positional Encodings
        features = features + self.pos_embed
        
        # Step D: Process through Transformer
        trans_out = self.transformer(features)  # Shape: [B, 49, 512]
        
        # Step E: Global Average Pooling over the sequence length
        pooled_out = trans_out.mean(dim=1)  # Shape: [B, 512]
        
        # Step F: Final Classification
        return self.classifier(pooled_out)

def execute_hybrid_pipeline():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = Hybrid_CNN_Transformer(num_classes=3)
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)
    
    # Utilizing our class weights to maintain strict penalty on minority class
    weights = class_weights_tensor.to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    
    # We use a lower learning rate because the CNN is pre-trained
    optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
    
    EPOCHS = 10 
    print(f"\n[INFO] Commencing Training Phase for Hybrid Masterpiece ({EPOCHS} Epochs)...")
    
    # NOTE: We reuse train_loader_vit because it already has the 224x224 resolution required
    for epoch in range(EPOCHS):
        start_time = time.time()
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader_vit:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        epoch_time = time.time() - start_time
        print(f"Epoch [{epoch+1}/{EPOCHS}] | Training Loss: {running_loss/len(train_loader_vit):.4f} | Time: {epoch_time:.1f}s")
        
    print("\n[INFO] Running Inference on Validation Set...")
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, labels in val_loader_vit:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
    classes = ["Normal", "Cardiomegaly", "Pneumonia/Opacity"]
    evaluator = ResearchEvaluator(model_name="Hybrid_CNN_Transformer")
    evaluator.evaluate_and_report(all_targets, all_preds, classes)

if __name__ == "__main__":
    execute_hybrid_pipeline()

STEP 8: HYBRID SOTA (CNN + TRANSFORMER FUSION)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 163MB/s] 



[INFO] Commencing Training Phase for Hybrid Masterpiece (10 Epochs)...
Epoch [1/10] | Training Loss: 0.9942 | Time: 165.7s
Epoch [2/10] | Training Loss: 0.7959 | Time: 163.3s
Epoch [3/10] | Training Loss: 0.6680 | Time: 162.9s
Epoch [4/10] | Training Loss: 0.5701 | Time: 163.5s
Epoch [5/10] | Training Loss: 0.4217 | Time: 163.0s
Epoch [6/10] | Training Loss: 0.3146 | Time: 162.7s
Epoch [7/10] | Training Loss: 0.2049 | Time: 161.9s
Epoch [8/10] | Training Loss: 0.1566 | Time: 162.8s
Epoch [9/10] | Training Loss: 0.1378 | Time: 162.6s
Epoch [10/10] | Training Loss: 0.1042 | Time: 162.2s

[INFO] Running Inference on Validation Set...

[INFO] Generating Research Metrics for Hybrid_CNN_Transformer...
Accuracy:  0.6610
Precision: 0.5868
Recall:    0.6610
F1-Score:  0.6036
[SUCCESS] Metrics dynamically appended to /kaggle/working/Comparative_Results.xlsx
[SUCCESS] High-Res Confusion Matrix saved to /kaggle/working/Hybrid_CNN_Transformer_Confusion_Matrix.png


### Phase 7: Overcoming the "Data Wall" (Multi-Label Engineering)
> **Observation:** SOTA models plateaued due to overlapping diseases (e.g., Cardiomegaly co-occurring with Pneumonia) creating mathematical contradictions under Softmax.
> **Solution:** Upgrading the pipeline to a Multi-Label Architecture using `BCEWithLogitsLoss` to output independent class probabilities.

In [2]:
import pandas as pd
import numpy as np
import os
import re
import torch
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings("ignore")

print("="*60)
print("SYSTEM REBOOT: FAST-TRACK DATA INITIALIZATION")
print("="*60)

# 1. Setup Directories
WORK_DIR = '/kaggle/working/'
os.makedirs(f'{WORK_DIR}data/processed', exist_ok=True)
CSV_IN_PATH = '/kaggle/input/datasets/sameernadeem66/xray-custom-code/processed_data/processed_data/master_dataset.csv'
CSV_OUT_PATH = f'{WORK_DIR}data/processed/classification_target.csv'
IMAGE_DIR = '/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized'

# 2. Extract Labels (If not already extracted)
if not os.path.exists(CSV_OUT_PATH):
    print("[INFO] Generating Classification Labels...")
    df = pd.read_csv(CSV_IN_PATH)
    
    def apply_rules(text):
        text = str(text).lower()
        if re.search(r'\b(cardiomegaly|heart is enlarged|enlarged heart)\b', text):
            return 1, "Cardiomegaly"
        elif re.search(r'\b(opacity|pneumonia|infiltrate|consolidation|effusion)\b', text):
            return 2, "Pneumonia/Opacity"
        else:
            return 0, "Normal"

    df['full_report'] = df['findings'].fillna('') + " " + df['impression'].fillna('')
    df[['class_id', 'class_name']] = df['full_report'].apply(apply_rules).tolist()
    final_df = df[['filename', 'class_id', 'class_name']]
    final_df.to_csv(CSV_OUT_PATH, index=False)
else:
    print("[INFO] Classification dataset already exists.")
    final_df = pd.read_csv(CSV_OUT_PATH)

# 3. Calculate Class Weights for strict minority class protection
print("[INFO] Recalculating Mathematical Class Weights...")
labels = final_df['class_id'].values
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print("\n[SUCCESS] Environment Ready for 30-Epoch Deep Training!")
print(f"GPUs Available: {torch.cuda.device_count()}")
for i, weight in enumerate(class_weights):
    print(f" -> Class {i} Penalty Weight: {weight:.4f}")

SYSTEM REBOOT: FAST-TRACK DATA INITIALIZATION
[INFO] Classification dataset already exists.
[INFO] Recalculating Mathematical Class Weights...

[SUCCESS] Environment Ready for 30-Epoch Deep Training!
GPUs Available: 2
 -> Class 0 Penalty Weight: 1.4523
 -> Class 1 Penalty Weight: 4.5892
 -> Class 2 Penalty Weight: 0.4777


In [3]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

print("="*60)
print("STEP 2: HAM SOTA TRAINING (30 EPOCHS + AMP + SCHEDULER)")
print("="*60)

# ---------------------------------------------------------
# 1. RE-ESTABLISH DATA PIPELINE (Required after restart)
# ---------------------------------------------------------
class XRayDataset(Dataset):
    def __init__(self, csv_file, image_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform
        available_images = set(os.listdir(self.image_dir))
        self.data = self.data[self.data['filename'].isin(available_images)].reset_index(drop=True)

    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        img_name = os.path.join(self.image_dir, self.data.iloc[idx, 0])
        image = Image.open(img_name).convert('RGB')
        label = int(self.data.iloc[idx, 1])
        if self.transform: image = self.transform(image)
        return image, label

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("[INFO] Loading Datasets into memory...")
dataset = XRayDataset('/kaggle/working/data/processed/classification_target.csv', 
                      '/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized', 
                      transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Using 64 batch size to maximize the Dual GPUs
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

# ---------------------------------------------------------
# 2. RE-ESTABLISH MLOPS EVALUATOR
# ---------------------------------------------------------
class ResearchEvaluator:
    def __init__(self, model_name, out_dir='/kaggle/working/'):
        self.model_name = model_name
        self.out_dir = out_dir
        self.excel_path = os.path.join(out_dir, 'Comparative_Results.xlsx')
        
    def evaluate_and_report(self, y_true, y_pred, classes):
        print(f"\n[INFO] Generating Final Metrics for {self.model_name}...")
        acc = accuracy_score(y_true, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
        
        print(f"Accuracy:  {acc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1-Score: {f1:.4f}")
        
        results_df = pd.DataFrame({'Model Architecture': [self.model_name], 'Accuracy': [acc], 'Precision': [precision], 'Recall': [recall], 'F1-Score': [f1]})
        if os.path.exists(self.excel_path):
            existing_df = pd.read_excel(self.excel_path)
            updated_df = pd.concat([existing_df, results_df], ignore_index=True)
            updated_df.to_excel(self.excel_path, index=False)
        else:
            results_df.to_excel(self.excel_path, index=False)
            
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
        plt.title(f'Confusion Matrix: {self.model_name} (30 Epochs)', fontsize=14, fontweight='bold')
        plot_path = os.path.join(self.out_dir, f'{self.model_name}_30Epochs_CM.png')
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()

# ---------------------------------------------------------
# 3. THE HAM ARCHITECTURE & ADVANCED TRAINING LOOP
# ---------------------------------------------------------
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        attention_map = self.conv(torch.cat([avg_out, max_out], dim=1))
        return x * self.sigmoid(attention_map)

class HAM_Network(nn.Module):
    def __init__(self, num_classes=3):
        super(HAM_Network, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.attention = SpatialAttention()
        self.high_level = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(128, num_classes))

    def forward(self, x):
        x = self.features(x)
        x = self.attention(x)
        x = torch.flatten(self.high_level(x), 1)
        return self.classifier(x)

def run_pro_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = nn.DataParallel(HAM_Network(num_classes=3)).to(device)
    
    # Re-using the weights calculated in the fast-track step
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
    
    # 10x Science: AdamW with weight decay + Cosine Annealing Scheduler
    optimizer = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-3)
    EPOCHS = 30
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    # 10x Science: Automatic Mixed Precision (AMP) Scaler
    scaler = torch.cuda.amp.GradScaler()
    
    print(f"\n[INFO] Starting Deep Training for {EPOCHS} Epochs...")
    print("[INFO] Active Sciences: Automatic Mixed Precision (AMP) | Cosine Annealing LR | Dual-GPU Sync")
    
    for epoch in range(EPOCHS):
        start_time = time.time()
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            
            # AMP Autocast Context
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
                
            # Scaled Backpropagation
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += loss.item()
            
        # Step the scheduler to smoothly decay the learning rate
        scheduler.step()
        
        epoch_time = time.time() - start_time
        current_lr = optimizer.param_groups[0]['lr']
        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss: {running_loss/len(train_loader):.4f} | LR: {current_lr:.6f} | Time: {epoch_time:.1f}s")
            
    print("\n[INFO] Training Complete. Evaluating SOTA Metrics...")
    model.eval()
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images.to(device))
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.numpy())
            
    evaluator = ResearchEvaluator(model_name="HAM_Network_30Ep")
    evaluator.evaluate_and_report(all_targets, all_preds, ["Normal", "Cardiomegaly", "Pneumonia"])

if __name__ == "__main__":
    run_pro_training()

STEP 2: HAM SOTA TRAINING (30 EPOCHS + AMP + SCHEDULER)
[INFO] Loading Datasets into memory...

[INFO] Starting Deep Training for 30 Epochs...
[INFO] Active Sciences: Automatic Mixed Precision (AMP) | Cosine Annealing LR | Dual-GPU Sync
Epoch [1/30] | Loss: 1.0724 | LR: 0.001995 | Time: 208.9s
Epoch [5/30] | Loss: 1.0192 | LR: 0.001866 | Time: 162.9s
Epoch [10/30] | Loss: 0.9885 | LR: 0.001500 | Time: 162.5s
Epoch [15/30] | Loss: 0.9616 | LR: 0.001000 | Time: 159.8s
Epoch [20/30] | Loss: 0.9395 | LR: 0.000501 | Time: 168.6s
Epoch [25/30] | Loss: 0.9194 | LR: 0.000135 | Time: 168.2s
Epoch [30/30] | Loss: 0.9100 | LR: 0.000001 | Time: 169.2s

[INFO] Training Complete. Evaluating SOTA Metrics...

[INFO] Generating Final Metrics for HAM_Network_30Ep...
Accuracy:  0.4226 | Precision: 0.6262 | Recall: 0.4226 | F1-Score: 0.4555


In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision.models import ViT_B_16_Weights
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import warnings
warnings.filterwarnings("ignore")

print("="*60)
print("STEP 3: VISION TRANSFORMER SOTA (30 EPOCHS + AMP)")
print("="*60)

# ---------------------------------------------------------
# 1. UPGRADE RESOLUTION FOR ViT (224x224 strictly required)
# ---------------------------------------------------------
vit_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("[INFO] Re-initializing Vision Dataset for High-Res 224x224...")
# Reusing the dataset class we defined in the previous cell
dataset_vit = XRayDataset(
    '/kaggle/working/data/processed/classification_target.csv', 
    '/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized', 
    vit_transform
)

train_size = int(0.8 * len(dataset_vit))
val_size = len(dataset_vit) - train_size
train_dataset_vit, val_dataset_vit = random_split(dataset_vit, [train_size, val_size])

# Slightly smaller batch size (32) because 224x224 images take more VRAM
train_loader_vit = DataLoader(train_dataset_vit, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader_vit = DataLoader(val_dataset_vit, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# ---------------------------------------------------------
# 2. THE VISION TRANSFORMER ARCHITECTURE
# ---------------------------------------------------------
class ViT_SOTA(nn.Module):
    def __init__(self, num_classes=3):
        super(ViT_SOTA, self).__init__()
        
        # Load Pre-trained weights (Transfer Learning Magic)
        self.vit = models.vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
        
        # Freeze lower layers to retain ImageNet knowledge
        for param in self.vit.parameters():
            param.requires_grad = False
            
        # Unfreeze the last 2 Transformer Encoders for deep medical fine-tuning
        for param in self.vit.encoder.layers[-2:].parameters():
            param.requires_grad = True
            
        # Re-engineer the classification head for our 3 Medical Classes
        self.vit.heads = nn.Sequential(
            nn.Linear(768, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.vit(x)

# ---------------------------------------------------------
# 3. HIGH-PERFORMANCE TRAINING LOOP
# ---------------------------------------------------------
def run_vit_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = nn.DataParallel(ViT_SOTA(num_classes=3)).to(device)
    
    # Utilizing strict class weighting
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
    
    # 10x Science: Lower learning rate for fine-tuning pre-trained models
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4, weight_decay=1e-4)
    
    EPOCHS = 30
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler()
    
    print(f"\n[INFO] Starting Deep Transfer Learning for {EPOCHS} Epochs...")
    
    for epoch in range(EPOCHS):
        start_time = time.time()
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader_vit:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += loss.item()
            
        scheduler.step()
        epoch_time = time.time() - start_time
        current_lr = optimizer.param_groups[0]['lr']
        
        # Logging every 5 epochs to keep terminal clean
        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss: {running_loss/len(train_loader_vit):.4f} | LR: {current_lr:.6f} | Time: {epoch_time:.1f}s")
            
    print("\n[INFO] Training Complete. Evaluating SOTA Metrics...")
    model.eval()
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for images, labels in val_loader_vit:
            outputs = model(images.to(device))
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.numpy())
            
    evaluator = ResearchEvaluator(model_name="ViT_Transformer_30Ep")
    evaluator.evaluate_and_report(all_targets, all_preds, ["Normal", "Cardiomegaly", "Pneumonia"])

if __name__ == "__main__":
    run_vit_training()

🚀 STEP 3: VISION TRANSFORMER SOTA (30 EPOCHS + AMP)
[INFO] Re-initializing Vision Dataset for High-Res 224x224...
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 208MB/s] 



[INFO] Starting Deep Transfer Learning for 30 Epochs...
Epoch [1/30] | Loss: 1.0008 | LR: 0.000299 | Time: 173.8s
Epoch [5/30] | Loss: 0.6292 | LR: 0.000280 | Time: 196.7s
Epoch [10/30] | Loss: 0.1885 | LR: 0.000225 | Time: 196.6s
Epoch [15/30] | Loss: 0.0407 | LR: 0.000150 | Time: 197.0s
Epoch [20/30] | Loss: 0.0104 | LR: 0.000076 | Time: 195.8s
Epoch [25/30] | Loss: 0.0022 | LR: 0.000021 | Time: 196.6s
Epoch [30/30] | Loss: 0.0023 | LR: 0.000001 | Time: 194.8s

[INFO] Training Complete. Evaluating SOTA Metrics...

[INFO] Generating Final Metrics for ViT_Transformer_30Ep...
Accuracy:  0.6161 | Precision: 0.5968 | Recall: 0.6161 | F1-Score: 0.6030


In [5]:
import pandas as pd
import re
import warnings
warnings.filterwarnings("ignore")

print("="*70)
print("DIAGNOSTIC ENGINE: DEEP DATA CONTRADICTION ANALYSIS")
print("="*70)

# 1. Load the original raw master dataset
csv_path = '/kaggle/input/datasets/sameernadeem66/xray-custom-code/processed_data/processed_data/master_dataset.csv'

try:
    df = pd.read_csv(csv_path)
    df['full_report'] = df['findings'].fillna('') + " " + df['impression'].fillna('')
    
    # 2. Define strict medical keywords
    cardio_keywords = r'\b(cardiomegaly|heart is enlarged|enlarged heart)\b'
    pneumonia_keywords = r'\b(opacity|pneumonia|infiltrate|consolidation|effusion)\b'
    
    # 3. Analyze the intersection of diseases
    df['has_cardio'] = df['full_report'].str.contains(cardio_keywords, case=False, regex=True)
    df['has_pneumonia'] = df['full_report'].str.contains(pneumonia_keywords, case=False, regex=True)
    
    total_records = len(df)
    only_cardio = len(df[df['has_cardio'] & ~df['has_pneumonia']])
    only_pneumonia = len(df[~df['has_cardio'] & df['has_pneumonia']])
    overlap_count = len(df[df['has_cardio'] & df['has_pneumonia']])
    normal_count = len(df[~df['has_cardio'] & ~df['has_pneumonia']])
    
    # 4. Report Generation
    print(f"[SYSTEM] Total Medical Reports Scanned: {total_records}\n")
    print(f"-> Pure Cardiomegaly Cases: {only_cardio}")
    print(f"-> Pure Pneumonia/Opacity Cases: {only_pneumonia}")
    print(f"-> Normal / Other Cases: {normal_count}")
    print(f"-> OVERLAPPING CASES (Both Diseases): {overlap_count}\n")
    
    print("="*70)
    print("ENGINEERING VERDICT:")
    if overlap_count > 0:
        error_percentage = (overlap_count / (only_cardio + overlap_count)) * 100
        print(f"Critical Data Flaw Detected. {overlap_count} records contain multiple diseases.")
        print(f"In our previous pipeline, {error_percentage:.1f}% of Cardiomegaly labels were corrupted")
        print("because they also contained Pneumonia, but the model was penalized for detecting it.")
        print("This mathematically capped our validation accuracy at ~65%.")
    else:
        print("No overlap detected. Data is cleanly separable.")
        
except Exception as e:
    print(f"[CRITICAL ERROR] Failed to load dataset. Check path. Details: {e}")

DIAGNOSTIC ENGINE: DEEP DATA CONTRADICTION ANALYSIS
[SYSTEM] Total Medical Reports Scanned: 6457

-> Pure Cardiomegaly Cases: 100
-> Pure Pneumonia/Opacity Cases: 4506
-> Normal / Other Cases: 1482
-> OVERLAPPING CASES (Both Diseases): 369

ENGINEERING VERDICT:
Critical Data Flaw Detected. 369 records contain multiple diseases.
In our previous pipeline, 78.7% of Cardiomegaly labels were corrupted
because they also contained Pneumonia, but the model was penalized for detecting it.
This mathematically capped our validation accuracy at ~65%.


### Phase 8: Empirical Evaluation & Telemetry
> Automated metric logging via a custom Object-Oriented `ResearchEvaluator`. Extracting Macro-F1, Precision, and Recall to empirically prove the "Accuracy Paradox" on text-mined clinical data.

In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision.models import ViT_B_16_Weights
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, multilabel_confusion_matrix
import warnings
warnings.filterwarnings("ignore")

print("="*70)
print("PHASE: 100x ELITE MULTI-LABEL ViT SOTA")
print("="*70)

WORK_DIR = '/kaggle/working/'
CSV_IN_PATH = '/kaggle/input/datasets/sameernadeem66/xray-custom-code/processed_data/processed_data/master_dataset.csv'
IMAGE_DIR = '/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized'

# 1. Multi-Label Data Engineering
print("[INFO] Engineering Multi-Label Target Matrix...")
df = pd.read_csv(CSV_IN_PATH)
df['full_report'] = df['findings'].fillna('') + " " + df['impression'].fillna('')

cardio_keywords = r'\b(cardiomegaly|heart is enlarged|enlarged heart)\b'
pneumonia_keywords = r'\b(opacity|pneumonia|infiltrate|consolidation|effusion)\b'

df['has_cardio'] = df['full_report'].str.contains(cardio_keywords, case=False, regex=True).astype(int)
df['has_pneumonia'] = df['full_report'].str.contains(pneumonia_keywords, case=False, regex=True).astype(int)

# Normal is strictly when both are 0
df['is_normal'] = ((df['has_cardio'] == 0) & (df['has_pneumonia'] == 0)).astype(int)

multi_label_df = df[['filename', 'is_normal', 'has_cardio', 'has_pneumonia']].copy()
print(f"[INFO] Multi-Label Dataset Generated. Total records: {len(multi_label_df)}")

# Calculate Positive Weights for BCEWithLogitsLoss to handle severe imbalance
pos_counts = multi_label_df[['is_normal', 'has_cardio', 'has_pneumonia']].sum().values
neg_counts = len(multi_label_df) - pos_counts
pos_weights_array = neg_counts / (pos_counts + 1e-5)
print(f"[INFO] Calculated Pos-Weights to force minority learning: {pos_weights_array}")

# 2. Advanced Vision Dataset
class MultiLabelXRayDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.data = dataframe
        self.image_dir = image_dir
        self.transform = transform
        available_images = set(os.listdir(self.image_dir))
        self.data = self.data[self.data['filename'].isin(available_images)].reset_index(drop=True)

    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        img_name = os.path.join(self.image_dir, self.data.iloc[idx]['filename'])
        image = Image.open(img_name).convert('RGB')
        
        # Extract the 3 labels as a Float Tensor for BCE Loss
        labels = self.data.iloc[idx][['is_normal', 'has_cardio', 'has_pneumonia']].values.astype(np.float32)
        labels_tensor = torch.tensor(labels)
        
        if self.transform: image = self.transform(image)
        return image, labels_tensor

vit_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset_multi = MultiLabelXRayDataset(multi_label_df, IMAGE_DIR, vit_transform)
train_size = int(0.8 * len(dataset_multi))
val_size = len(dataset_multi) - train_size
train_dataset, val_dataset = random_split(dataset_multi, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# 3. Multi-Label ViT Architecture
class ViT_MultiLabel(nn.Module):
    def __init__(self, num_classes=3):
        super(ViT_MultiLabel, self).__init__()
        self.vit = models.vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
        
        for param in self.vit.parameters():
            param.requires_grad = False
        for param in self.vit.encoder.layers[-2:].parameters():
            param.requires_grad = True
            
        self.vit.heads = nn.Sequential(
            nn.Linear(768, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
            # No Softmax or Sigmoid here. BCEWithLogitsLoss handles it mathematically safely.
        )

    def forward(self, x):
        return self.vit(x)

def run_multilabel_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = nn.DataParallel(ViT_MultiLabel(num_classes=3)).to(device)
    
    # Injecting the calculated pos_weights to the loss function
    pos_weight_tensor = torch.tensor(pos_weights_array, dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
    
    EPOCHS = 30
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.cuda.amp.GradScaler()
    
    best_f1 = 0.0
    best_model_path = os.path.join(WORK_DIR, "best_multilabel_vit.pth")
    
    print(f"\n[INFO] Commencing 30-Epoch Multi-Label Training...")
    
    for epoch in range(EPOCHS):
        start_time = time.time()
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            
        scheduler.step()
        
        # Validation Phase
        model.eval()
        all_preds = []
        all_targets = []
        with torch.no_grad():
            for images, labels in val_loader:
                outputs = model(images.to(device))
                # Apply Sigmoid to convert logits to probabilities, then threshold at 0.5
                probs = torch.sigmoid(outputs)
                preds = (probs > 0.5).float()
                
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(labels.cpu().numpy())
                
        all_preds = np.array(all_preds)
        all_targets = np.array(all_targets)
        
        # In multi-label, macro F1 is the absolute best indicator of learning across all classes
        macro_f1 = precision_recall_fscore_support(all_targets, all_preds, average='macro', zero_division=0)[2]
        subset_accuracy = accuracy_score(all_targets, all_preds)
        
        epoch_time = time.time() - start_time
        print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss: {running_loss/len(train_loader):.4f} | Subset Acc: {subset_accuracy:.4f} | Macro F1: {macro_f1:.4f} | Time: {epoch_time:.1f}s")
        
        if macro_f1 > best_f1:
            best_f1 = macro_f1
            torch.save(model.state_dict(), best_model_path)
            print(f" -> [CHECKPOINT] New Best Macro F1 Hit: {macro_f1:.4f}")

if __name__ == "__main__":
    run_multilabel_training()